# ASSIGNMENT 2
## Exploring Text Generation Using GPT-2

 Generative AI  

 Lakhan Singh  


### Objective

The objective of this assignment is to understand how different text-generation parameters influence the output produced by a pretrained GPT-2 language model.

This notebook uses GPT-2 to perform five controlled generation experiments, compare the effects of temperature, top-k, top-p and repetition penalty, and generate three versions of a custom prompt.

## 1. Assignment Requirements

This notebook follows the assignment requirements:

- Use a pretrained GPT-2 model from Hugging Face Transformers.
- Use one fixed opening sentence for five experiments.
- Change one generation parameter at a time.
- Record the parameter, value and generated output.
- Analyze predictability, creativity, randomness, coherence and repetition.
- Create a custom prompt and generate three versions.
- Include a comparison table and brief analysis.
- Include an optional high-temperature experiment.

## 2. Install Required Libraries

The `transformers` library provides the pretrained GPT-2 model and tokenizer. PyTorch runs the model, while Pandas organizes the experiment results.

In [1]:
!pip install -q transformers torch pandas matplotlib

## 3. Import Required Libraries

We import GPT-2 and its tokenizer from Hugging Face, PyTorch for inference, Pandas for tables, and Matplotlib for simple visualizations.

In [2]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import torch
import pandas as pd
import matplotlib.pyplot as plt

print("Libraries imported successfully.")

Libraries imported successfully.


## 4. Load the Pretrained GPT-2 Model

GPT-2 is a pretrained autoregressive language model. It predicts probable next tokens based on the text that has already been provided.

The model is not trained from scratch in this assignment; the pretrained `gpt2` checkpoint is loaded from Hugging Face.

In [3]:
model_name = "gpt2"

tokenizer = GPT2Tokenizer.from_pretrained(model_name)
model = GPT2LMHeadModel.from_pretrained(model_name)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id
model.eval()

print("GPT-2 loaded successfully.")
print("Model:", model_name)
print("Number of parameters:", f"{model.num_parameters():,}")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

GPT-2 loaded successfully.
Model: gpt2
Number of parameters: 124,439,808


### How GPT-2 Generates Text

The basic process is:

**Prompt → Tokenizer → GPT-2 → Next-token probabilities → Sampling → Generated text**

GPT-2 repeatedly predicts the next token and appends it to the sequence.

## 5. Original Prompt

The same opening sentence is used for all five main experiments. Keeping the prompt fixed makes the comparison controlled because changes in the output can be related to the generation settings rather than to a different input.

**Original Prompt:**

> Artificial intelligence will change the way people work because

In [4]:
prompt = "Artificial intelligence will change the way people work because"
print("Original Prompt:")
print(prompt)

Original Prompt:
Artificial intelligence will change the way people work because


## 6. Generation Parameters

| Parameter | Purpose |
|---|---|
| **Temperature** | Controls the randomness of token selection. Lower values are generally more predictable; higher values are generally more random. |
| **Top-k** | Restricts sampling to the k most probable next tokens. |
| **Top-p** | Restricts sampling to a probability-based candidate set. |
| **Repetition penalty** | Penalizes previously used tokens and can reduce repetition. |

For the five main experiments, the prompt and random seed are kept fixed and only the selected parameter is changed.

## 7. Reusable Text-Generation Function

This function tokenizes the prompt, calls GPT-2, and converts the generated token IDs back into readable text. A fixed seed is used for the main experiments so that the parameter comparison is more controlled.

In [5]:
def generate_text(
    prompt,
    temperature=1.0,
    top_k=50,
    top_p=1.0,
    repetition_penalty=1.0,
    seed=42,
    max_new_tokens=80
):
    torch.manual_seed(seed)
    inputs = tokenizer(prompt, return_tensors="pt")

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=top_k,
            top_p=top_p,
            repetition_penalty=repetition_penalty,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

## 8. Experiment 1 — Baseline

The baseline uses the reference settings: temperature 1.0, top-k 50, top-p 1.0 and repetition penalty 1.0. The other experiments will change one parameter at a time.

In [6]:
baseline_output = generate_text(prompt, temperature=1.0, top_k=50, top_p=1.0, repetition_penalty=1.0, seed=42)
print("Experiment 1 — Baseline")
print("=" * 80)
print(baseline_output)

Experiment 1 — Baseline
Artificial intelligence will change the way people work because more and more work is done in AI. If you are already using machine learning and machine learning for research or research needs, the AI that you create will become what you create. The AI that you create will be what will give you the tools you need to have good experiences and use the results (if you take the money) to improve your business prospects. But if you leave your business in business


### Observation — Baseline

Use this output as the reference. Compare the other experiments for predictability, variety, coherence and repetition.

## 9. Experiment 2 — Temperature = 0.3

**Parameter changed:** Temperature  
**Value:** 0.3

Only temperature is changed. A lower temperature generally makes token selection more conservative and predictable.

In [7]:
temperature_output = generate_text(prompt, temperature=0.3, top_k=50, top_p=1.0, repetition_penalty=1.0, seed=42)
print("Experiment 2 — Temperature = 0.3")
print("=" * 80)
print(temperature_output)

Experiment 2 — Temperature = 0.3
Artificial intelligence will change the way people work because it will be able to do things that are not possible in the past.

It's a good thing that we're not living in a dystopian future. We're living in a world where people are constantly being told to do things that are not possible in the past.

I think that's a good thing.

I think that's a good thing.

I think that


### Observation — Temperature = 0.3

Compare this output with the baseline. Look for whether the continuation is more predictable, focused or repetitive and whether it appears less varied.

## 10. Experiment 3 — Top-k = 20

**Parameter changed:** Top-k  
**Value:** 20

Top-k limits the next-token candidates to the 20 most probable choices. A smaller value can make generation more controlled.

In [8]:
top_k_output = generate_text(prompt, temperature=1.0, top_k=20, top_p=1.0, repetition_penalty=1.0, seed=42)
print("Experiment 3 — Top-k = 20")
print("=" * 80)
print(top_k_output)

Experiment 3 — Top-k = 20
Artificial intelligence will change the way people work because more and more work is done in AI. If you are already using machine learning and machine learning for research or research needs, the AI that you create will become what you create. The AI that you create will be what will give you the tools you need to develop and market smart machines. The AI that you sell to your customers will be the tools you use to make their business more successful in your


### Observation — Top-k = 20

Compare the variety and predictability with the baseline. Check whether restricting the candidate set changes coherence or word choice.

## 11. Experiment 4 — Top-p = 0.8

**Parameter changed:** Top-p  
**Value:** 0.8

Top-p, or nucleus sampling, selects from a probability-based group of candidate tokens. The group can change at each generation step.

In [9]:
top_p_output = generate_text(prompt, temperature=1.0, top_k=50, top_p=0.8, repetition_penalty=1.0, seed=42)
print("Experiment 4 — Top-p = 0.8")
print("=" * 80)
print(top_p_output)

Experiment 4 — Top-p = 0.8
Artificial intelligence will change the way people work because we will be able to create algorithms that will make our jobs more productive and less stressful."

So far, there is little research into artificial intelligence, and its use is limited. A recent study of 15,000 employees at a German government agency found that artificial intelligence had not improved the way people work, because the work was "not as enjoyable."

But experts say that more research on


### Observation — Top-p = 0.8

Compare the output with the baseline. Look at diversity, coherence and how restricted or varied the continuation appears.

## 12. Experiment 5 — Repetition Penalty = 1.5

**Parameter changed:** Repetition penalty  
**Value:** 1.5

A repetition penalty greater than 1 discourages the model from repeatedly selecting tokens that have already appeared.

In [10]:
repetition_output = generate_text(prompt, temperature=1.0, top_k=50, top_p=1.0, repetition_penalty=1.5, seed=42)
print("Experiment 5 — Repetition Penalty = 1.5")
print("=" * 80)
print(repetition_output)

Experiment 5 — Repetition Penalty = 1.5
Artificial intelligence will change the way people work because more and better technology is required in order for us to achieve jobs that are hard or repetitive.



### Observation — Repetition Penalty = 1.5

Compare this result with the baseline. Focus especially on whether repeated words or phrases are reduced and whether fluency changes.

## 13. Five-Experiment Comparison Table

This table records the original prompt, changed parameter, value and actual output for all five experiments.

In [11]:
experiment_results = pd.DataFrame([
    ["1 - Baseline", prompt, "Baseline", "T=1.0, K=50, P=1.0, RP=1.0", baseline_output],
    ["2 - Temperature", prompt, "Temperature", "0.3", temperature_output],
    ["3 - Top-k", prompt, "Top-k", "20", top_k_output],
    ["4 - Top-p", prompt, "Top-p", "0.8", top_p_output],
    ["5 - Repetition Penalty", prompt, "Repetition Penalty", "1.5", repetition_output]
], columns=["Experiment", "Original Prompt", "Parameter Changed", "Value", "Generated Output"])

pd.set_option("display.max_colwidth", 300)
experiment_results

,Experiment,Original Prompt,Parameter Changed,Value,Generated Output
0,1 - Baseline,Artificial intelligence will change the way people work because,Baseline,"T=1.0, K=50, P=1.0, RP=1.0","Artificial intelligence will change the way people work because more and more work is done in AI. If you are already using machine learning and machine learning for research or research needs, the AI that you create will become what you create. The AI that you create will be what will give you t..."
1,2 - Temperature,Artificial intelligence will change the way people work because,Temperature,0.3,Artificial intelligence will change the way people work because it will be able to do things that are not possible in the past.\n\nIt's a good thing that we're not living in a dystopian future. We're living in a world where people are constantly being told to do things that are not possible in t...
2,3 - Top-k,Artificial intelligence will change the way people work because,Top-k,20,"Artificial intelligence will change the way people work because more and more work is done in AI. If you are already using machine learning and machine learning for research or research needs, the AI that you create will become what you create. The AI that you create will be what will give you t..."
3,4 - Top-p,Artificial intelligence will change the way people work because,Top-p,0.8,"Artificial intelligence will change the way people work because we will be able to create algorithms that will make our jobs more productive and less stressful.""\n\nSo far, there is little research into artificial intelligence, and its use is limited. A recent study of 15,000 employees at a Germ..."
4,5 - Repetition Penalty,Artificial intelligence will change the way people work because,Repetition Penalty,1.5,Artificial intelligence will change the way people work because more and better technology is required in order for us to achieve jobs that are hard or repetitive.\n


## 14. Analysis of the Five Experiments

After running the experiments, use the actual outputs in the table to decide:

- Which setting produced the most repetitive or predictable text?
- Which setting produced the most creative text while remaining coherent?
- Which setting produced the most random or incoherent text?
- Did the repetition penalty reduce repeated wording?

### Important

Do not assume that every run will look exactly the same. GPT-2 uses sampling, so the actual output should be the basis for your observations.

## 15. General Parameter Summary

The expected general behavior is:

- **Lower temperature:** usually more predictable and conservative.
- **Higher temperature:** usually more varied and random; very high values can reduce coherence.
- **Lower top-k:** fewer candidate tokens and more restricted generation.
- **Lower top-p:** smaller probability-based candidate set and more controlled generation.
- **Higher repetition penalty:** stronger discouragement of repeated tokens.

These are general tendencies; the observations in the report should be based on the outputs actually generated.

In [12]:
parameter_summary = pd.DataFrame({
    "Parameter": ["Temperature", "Top-k", "Top-p", "Repetition Penalty"],
    "Main Purpose": [
        "Controls randomness of token selection",
        "Limits selection to the k most probable tokens",
        "Limits selection using cumulative probability",
        "Reduces repeated token usage"
    ],
    "Typical Effect": [
        "Lower = more predictable; higher = more random",
        "Lower = more restricted and controlled",
        "Lower = more restricted; higher = more diverse",
        "Higher = less repetition"
    ]
})
parameter_summary

,Parameter,Main Purpose,Typical Effect
0,Temperature,Controls randomness of token selection,Lower = more predictable; higher = more random
1,Top-k,Limits selection to the k most probable tokens,Lower = more restricted and controlled
2,Top-p,Limits selection using cumulative probability,Lower = more restricted; higher = more diverse
3,Repetition Penalty,Reduces repeated token usage,Higher = less repetition


## 16. Custom Prompt

For the second part, the following story-opening prompt is used:

> **The young engineer opened the mysterious box and discovered**

The same prompt is used for all three versions. Different random seeds allow different sampled continuations.

In [13]:
custom_prompt = "The young engineer opened the mysterious box and discovered"
print("Custom Prompt:")
print(custom_prompt)

Custom Prompt:
The young engineer opened the mysterious box and discovered


## 17. Custom Generation — Version 1

The first version uses moderate sampling settings.

In [14]:
custom_output_1 = generate_text(custom_prompt, temperature=0.7, top_k=50, top_p=0.9, repetition_penalty=1.1, seed=101)
print(custom_output_1)

The young engineer opened the mysterious box and discovered a small, large bottle of wine.
"Well I hope it's not too much for my taste buds to swallow," he said with his hands on his hips as they waited in line at our table. "And if you want some more help then please take them home."
We had just started chatting about how we could improve this app by letting us share photos via social media using photo sharing


## 18. Custom Generation — Version 2

The same prompt and settings are used with a different seed, allowing a different sampled continuation.

In [15]:
custom_output_2 = generate_text(custom_prompt, temperature=0.7, top_k=50, top_p=0.9, repetition_penalty=1.1, seed=202)
print(custom_output_2)

The young engineer opened the mysterious box and discovered that he was in possession of a key to his world. He then left, leaving only what little information remained: The message had been sent by an ancient demon called "the Lancer".
What did this mean for humanity? How would they survive against such monsters who were seemingly invincible before their own eyes? All we know is there are three main reasons why humans cannot defeat them...

A very


## 19. Custom Generation — Version 3

A third version is generated from exactly the same prompt and generation settings, using another random seed.

In [16]:
custom_output_3 = generate_text(custom_prompt, temperature=0.7, top_k=50, top_p=0.9, repetition_penalty=1.1, seed=303)
print(custom_output_3)

The young engineer opened the mysterious box and discovered that he had been given a copy of this book. The man who wrote it was very old, but now looked like an angel in his youth: "I am from heaven! I have just returned to my place at once."

"It is not true," said Professor Pius; "but there are many things which you may learn by studying these books!" He smiled with joy as they


## 20. Custom Generation Comparison

The table below records the three generated versions from the same custom prompt.

In [17]:
custom_results = pd.DataFrame([
    ["Version 1", custom_prompt, 0.7, 50, 0.9, 1.1, custom_output_1],
    ["Version 2", custom_prompt, 0.7, 50, 0.9, 1.1, custom_output_2],
    ["Version 3", custom_prompt, 0.7, 50, 0.9, 1.1, custom_output_3]
], columns=["Version", "Prompt", "Temperature", "Top-k", "Top-p", "Repetition Penalty", "Generated Output"])

pd.set_option("display.max_colwidth", 300)
custom_results

,Version,Prompt,Temperature,Top-k,Top-p,Repetition Penalty,Generated Output
0,Version 1,The young engineer opened the mysterious box and discovered,0.7,50,0.9,1.1,"The young engineer opened the mysterious box and discovered a small, large bottle of wine.\n""Well I hope it's not too much for my taste buds to swallow,"" he said with his hands on his hips as they waited in line at our table. ""And if you want some more help then please take them home.""\nWe had j..."
1,Version 2,The young engineer opened the mysterious box and discovered,0.7,50,0.9,1.1,"The young engineer opened the mysterious box and discovered that he was in possession of a key to his world. He then left, leaving only what little information remained: The message had been sent by an ancient demon called ""the Lancer"".\nWhat did this mean for humanity? How would they survive ag..."
2,Version 3,The young engineer opened the mysterious box and discovered,0.7,50,0.9,1.1,"The young engineer opened the mysterious box and discovered that he had been given a copy of this book. The man who wrote it was very old, but now looked like an angel in his youth: ""I am from heaven! I have just returned to my place at once.""\n\n""It is not true,"" said Professor Pius; ""but there..."


## 21. Custom Prompt Observations

Compare the three versions and discuss:

- Whether the story continued differently each time.
- Which version was the most coherent.
- Which version was the most interesting or creative.
- Whether any version repeated words or ideas.

### Example observation

> The three versions started from the same prompt but produced different continuations. This demonstrates that sampled language generation can produce multiple possible continuations from the same starting text.

## 22. Additional Activity — High Temperature

The assignment optionally asks for an unusual or meaningless output at a high temperature. The following experiment uses temperature 1.8.

If the output is unusually incoherent, take a screenshot of it and include the screenshot in the submitted report. If it remains coherent, simply state that the high-temperature output remained reasonably coherent.

In [18]:
high_temperature_output = generate_text(prompt, temperature=1.8, top_k=50, top_p=1.0, repetition_penalty=1.0, seed=999)
print("High-Temperature Experiment — Temperature = 1.8")
print("=" * 80)
print(high_temperature_output)

High-Temperature Experiment — Temperature = 1.8
Artificial intelligence will change the way people work because the computer has tools (AI or machines) to analyse data at an atomic speed. Artificial intelligence will now take the top-performing firms to court in industries where a single machine produces the work of human intelligence more and more in our day. These decisions take time to make. Most work with machines, a huge amount at that, is a very slow activity, at about 2½ percent per minute.


### Why High Temperature Can Produce Unusual Text

Higher temperature increases the randomness of token sampling. This makes lower-probability tokens more likely to be selected. The result can be more varied and surprising, but at sufficiently high values it can also become less coherent or contain unusual word combinations.

## 23. Overall Analysis

The five experiments demonstrate that generation parameters provide different ways to control GPT-2 output. Temperature primarily changes randomness, top-k restricts the number of high-probability candidates, top-p restricts the candidate set using cumulative probability, and repetition penalty discourages repeated tokens. Lower-randomness settings generally produce more controlled text, while higher randomness can provide more variation but may reduce coherence. The best settings depend on whether predictability, creativity, diversity or reduced repetition is the main goal.



GPT-2 generates text by learning language patterns during pretraining and predicting likely next tokens from the current context. Generation parameters influence how those predicted tokens are selected. Lower randomness generally makes the output more predictable, while higher randomness allows more variation. Top-k and top-p restrict the possible token choices, while repetition penalty reduces the tendency to reuse tokens. This assignment showed how small changes in generation parameters can significantly affect the style and quality of generated text.

## 25. Conclusion

This assignment explored text generation using a pretrained GPT-2 model. Five controlled experiments showed how temperature, top-k, top-p and repetition penalty influence generated text. Three custom generations demonstrated that the same prompt can produce different continuations when sampling is used. Overall, the experiments show that generation parameters are important tools for balancing predictability, creativity, randomness, coherence and repetition in language-model output.